In [2]:
# 1. Install dependencies
# pip install prophet pandas scikit-learn joblib plotly

import pandas as pd
from prophet import Prophet
import joblib
import os
from datetime import datetime

c:\Users\2006r\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
df_raw = pd.read_csv('./data/expenses_rows.csv')  # Replace with your actual data file path

In [38]:
df_raw.drop(columns=['id', 'user_id', 'description', "created_at"], inplace=True)
df_raw.replace({'category_id': {
    "113225e0-523b-4f3c-af5e-3b1bfc17b202": "Shopping",
    "3e320471-accc-4b94-b4e6-1a2d853d51a1": "Entertainment",
    "9d500dc7-0fc5-47b5-bf3b-1d5f57982793": "transport",
    "a4615ed1-a622-4ba1-9527-99a2f99ece5b": "Miscellaneous",
    "dfe79feb-8e63-4016-9994-706f81570549": "Food",
    "f3bccc52-435d-4fec-bb68-8ce687e665eb" : "Bills"
}}, inplace=True)
df_raw.rename(columns={'category_id': 'category'}, inplace=True)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw

,category,amount,date
0,Bills,78.0,2025-12-07
1,Food,265.0,2026-01-29
2,transport,100.0,2026-03-27
3,transport,100.0,2026-01-30
4,transport,100.0,2025-11-28
...,...,...,...
448,transport,47.0,2026-01-09
449,transport,100.0,2026-03-09
450,transport,100.0,2026-02-03
451,Food,250.0,2025-11-08


In [39]:
df = df_raw.groupby(['category', pd.Grouper(key='date', freq='MS')])['amount'].sum().reset_index()
df.columns = ['category', 'ds', 'y']

In [40]:
# 3. Train one model per category & save
MODELS_DIR = "models/"
os.makedirs(MODELS_DIR, exist_ok=True)

categories = df['category'].unique()

for cat in categories:
    cat_df = df[df['category'] == cat][['ds', 'y']].reset_index(drop=True)
    
    model = Prophet(
        yearly_seasonality=False,  # not enough data yet
        weekly_seasonality=False,  # monthly data, not needed
        daily_seasonality=False,
        seasonality_mode='additive'
    )
    model.fit(cat_df)
    
    # Save model
    joblib.dump(model, f"{MODELS_DIR}/{cat}_model.pkl")
    print(f"✓ Trained and saved model for: {cat}")

02:07:36 - cmdstanpy - INFO - Chain [1] start processing
02:07:38 - cmdstanpy - INFO - Chain [1] done processing
02:07:38 - cmdstanpy - INFO - Chain [1] start processing
02:07:38 - cmdstanpy - INFO - Chain [1] done processing


✓ Trained and saved model for: Bills
✓ Trained and saved model for: Entertainment


02:07:38 - cmdstanpy - INFO - Chain [1] start processing
02:07:38 - cmdstanpy - INFO - Chain [1] done processing
02:07:38 - cmdstanpy - INFO - Chain [1] start processing
02:07:38 - cmdstanpy - INFO - Chain [1] done processing


✓ Trained and saved model for: Food
✓ Trained and saved model for: Miscellaneous


02:07:38 - cmdstanpy - INFO - Chain [1] start processing
02:07:38 - cmdstanpy - INFO - Chain [1] done processing
02:07:38 - cmdstanpy - INFO - Chain [1] start processing
02:07:38 - cmdstanpy - INFO - Chain [1] done processing


✓ Trained and saved model for: Shopping
✓ Trained and saved model for: transport
